# 04 — Diagnóstico de cambios estructurales

> XLE es el caso principal. El objetivo es separar shocks residuales aislados de errores que persisten y mantienen relación con los factores; las alertas son diagnósticas, no pruebas confirmatorias de un régimen.

## <b>Modelos congelados y reproducción</b>

Se reproducen rolling OLS de 252 sesiones y Kalman con $Q/R$ fijos de la notebook 02. Cada atribución en $t$ usa coeficientes disponibles hasta $t-1$ y factores realizados en $t$; la evaluación es 2022–2026.

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots
from IPython.display import HTML, display

TARGETS = ('QQQ', 'XLE', 'XLF')
SYMBOLS = ('SPY', 'QQQ', 'IWM', 'IWD', 'IWF', 'XLE', 'XLF')
FACTORS = ('market', 'size', 'growth_value')
RESEARCH_START, RESEARCH_END = date(2018, 1, 2), date(2026, 11, 17)
CALIBRATION_END, EVALUATION_START = date(2021, 12, 31), date(2022, 1, 1)
ROLLING_WINDOW, DIAGNOSTIC_WINDOW = 252, 21

def project_root(start):
    for parent in (start.resolve(), *start.resolve().parents):
        direct = parent / 'Momentum_systematic_strategies'
        if (direct / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return direct
        if (parent / 'data' / 'gold' / 'daily_series' / 'etf').is_dir():
            return parent
    raise FileNotFoundError('No se encontró data/gold/daily_series/etf.')

GOLD = project_root(Path.cwd()) / 'data' / 'gold' / 'daily_series' / 'etf'
frames = {symbol: pl.read_parquet(GOLD / f'{symbol}.parquet').with_columns(pl.col('session_date').cast(pl.Date)).filter(
    pl.col('session_date').is_between(RESEARCH_START, RESEARCH_END, closed='both')).sort('session_date') for symbol in SYMBOLS}
prices = frames['SPY'].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias('SPY'))
for symbol in SYMBOLS[1:]:
    prices = prices.join(frames[symbol].select(pl.col('session_date').alias('date'), pl.col('adjusted_close').alias(symbol)), on='date', how='inner')
returns = prices.sort('date').with_columns([pl.col(symbol).log().diff().alias(f'{symbol}_ret') for symbol in SYMBOLS]).drop_nulls()
model_panel = returns.select('date', *[f'{asset}_ret' for asset in TARGETS], 'SPY_ret', 'IWM_ret', 'IWF_ret', 'IWD_ret').with_columns(
    pl.col('SPY_ret').alias('market'), (pl.col('IWM_ret') - pl.col('SPY_ret')).alias('size'),
    (pl.col('IWF_ret') - pl.col('IWD_ret')).alias('growth_value')).select('date', *[f'{asset}_ret' for asset in TARGETS], *FACTORS).drop_nulls()
dates, factor_values = model_panel.get_column('date').to_list(), model_panel.select(*FACTORS).to_numpy()
evaluation_index = next(index for index, value in enumerate(dates) if value >= EVALUATION_START)
assert dates[evaluation_index - 1] <= CALIBRATION_END

In [2]:
ols_rows, calibration = [], {}
for asset in TARGETS:
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    history, residual_history = [], []
    for index in range(ROLLING_WINDOW, len(actual)):
        design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[index - ROLLING_WINDOW:index]))
        beta = np.linalg.lstsq(design, actual[index - ROLLING_WINDOW:index], rcond=None)[0]
        predicted = np.r_[1.0, factor_values[index]] @ beta
        residual = actual[index] - predicted
        ols_rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                         'alpha': beta[0], 'beta_market': beta[1], 'beta_size': beta[2], 'beta_growth_value': beta[3],
                         'actual_return': actual[index], 'predicted_return': predicted, 'residual_return': residual})
        if dates[index] <= CALIBRATION_END:
            history.append(beta)
            residual_history.append(residual)
    history = np.asarray(history)
    r = float(np.var(residual_history, ddof=1))
    last_design = np.column_stack((np.ones(ROLLING_WINDOW), factor_values[evaluation_index - ROLLING_WINDOW:evaluation_index]))
    calibration[asset] = {'Q': np.diag(np.var(np.diff(history, axis=0), axis=0, ddof=1)), 'R': r,
                          'theta': history[-1], 'P': r * np.linalg.pinv(last_design.T @ last_design)}
ols = pl.DataFrame(ols_rows)

def run_kalman(asset):
    actual = model_panel.get_column(f'{asset}_ret').to_numpy()
    theta, covariance = calibration[asset]['theta'].copy(), calibration[asset]['P'].copy()
    rows = []
    for index in range(evaluation_index, len(actual)):
        x = np.r_[1.0, factor_values[index]]
        theta_prior, covariance_prior = theta, covariance + calibration[asset]['Q']
        predicted, error = x @ theta_prior, actual[index] - x @ theta_prior
        gain = covariance_prior @ x / float(x @ covariance_prior @ x + calibration[asset]['R'])
        theta = theta_prior + gain * error
        update = np.eye(4) - np.outer(gain, x)
        covariance = update @ covariance_prior @ update.T + np.outer(gain, gain) * calibration[asset]['R']
        rows.append({'date': dates[index], 'beta_asof_date': dates[index - 1], 'asset': asset,
                     'alpha': theta_prior[0], 'beta_market': theta_prior[1], 'beta_size': theta_prior[2],
                     'beta_growth_value': theta_prior[3], 'actual_return': actual[index],
                     'predicted_return': predicted, 'residual_return': error})
    return pl.DataFrame(rows)

kalman = pl.concat([run_kalman(asset) for asset in TARGETS])
ols_evaluation = ols.filter(pl.col('date') >= EVALUATION_START)
reference_r2 = {'QQQ': (0.9686, 0.9708), 'XLE': (0.3164, 0.3351), 'XLF': (0.7725, 0.7792)}
reproduction_rows = []
for asset in TARGETS:
    values = []
    for frame in (ols_evaluation, kalman):
        sample = frame.filter(pl.col('asset') == asset)
        actual, error = sample.get_column('actual_return').to_numpy(), sample.get_column('residual_return').to_numpy()
        values.append(1 - (error ** 2).sum() / ((actual - actual.mean()) ** 2).sum())
    reproduction_rows.append({'asset': asset, 'ols_r2': values[0], 'kalman_r2': values[1],
                             'reference_ols_r2': reference_r2[asset][0], 'reference_kalman_r2': reference_r2[asset][1]})
reproduction = pl.DataFrame(reproduction_rows).with_columns(pl.all().exclude('asset').round(4))
if any(abs(row['ols_r2'] - row['reference_ols_r2']) > 0.0001 or abs(row['kalman_r2'] - row['reference_kalman_r2']) > 0.0001 for row in reproduction_rows):
    raise AssertionError('Los modelos congelados no reproducen la notebook 02.')
common_observations = ols_evaluation.select('date', 'asset').join(kalman.select('date', 'asset'), on=['date', 'asset']).height
reproduction, common_observations

(shape: (3, 5)
 ┌───────┬────────┬───────────┬──────────────────┬─────────────────────┐
 │ asset ┆ ols_r2 ┆ kalman_r2 ┆ reference_ols_r2 ┆ reference_kalman_r2 │
 │ ---   ┆ ---    ┆ ---       ┆ ---              ┆ ---                 │
 │ str   ┆ f64    ┆ f64       ┆ f64              ┆ f64                 │
 ╞═══════╪════════╪═══════════╪══════════════════╪═════════════════════╡
 │ QQQ   ┆ 0.9686 ┆ 0.9708    ┆ 0.9686           ┆ 0.9708              │
 │ XLE   ┆ 0.3164 ┆ 0.3351    ┆ 0.3164           ┆ 0.3351              │
 │ XLF   ┆ 0.7725 ┆ 0.7792    ┆ 0.7725           ┆ 0.7792              │
 └───────┴────────┴───────────┴──────────────────┴─────────────────────┘,
 3672)

## <b>Shocks residuales e inestabilidad factorial</b>

Un shock usa el percentil 99 de $|e_t|$ de OLS antes de 2022. Para cada factor se calcula $m_{k,t} = (1/21) \sum_{j=0}^{20}F_{k,t-j}e_{t-j}$. Sus umbrales son el percentil 97.5 de $|m_{k,t}|$ previo a 2022. Un residual grande puede ser transitorio; una alerta factorial indica relación persistente entre error y factor, pero también puede reflejar factores omitidos.

In [3]:
xle_ols_all = ols.filter(pl.col('asset') == 'XLE').sort('date')
xle_factors = xle_ols_all.join(model_panel.select('date', *FACTORS), on='date', how='inner').sort('date')
calibration_mask = np.asarray([value <= CALIBRATION_END for value in xle_factors.get_column('date').to_list()])
errors = xle_factors.get_column('residual_return').to_numpy()
factor_matrix = xle_factors.select(*FACTORS).to_numpy()
shock_threshold = np.quantile(np.abs(errors[calibration_mask]), .99)
products = factor_matrix * errors[:, None]
rolling_product = np.full_like(products, np.nan)
for index in range(DIAGNOSTIC_WINDOW - 1, len(errors)):
    rolling_product[index] = products[index - DIAGNOSTIC_WINDOW + 1:index + 1].mean(axis=0)
factor_thresholds = np.quantile(np.abs(rolling_product[calibration_mask & np.isfinite(rolling_product).all(axis=1)]), .975, axis=0)
factor_score = np.abs(rolling_product) / factor_thresholds
evaluation_mask = np.asarray([value >= EVALUATION_START for value in xle_factors.get_column('date').to_list()])
diagnostic = xle_factors.with_columns([
    pl.Series('shock_alert', np.abs(errors) > shock_threshold),
    pl.Series('MKT_score', factor_score[:, 0]), pl.Series('SIZE_score', factor_score[:, 1]), pl.Series('GV_score', factor_score[:, 2]),
]).filter(pl.Series(evaluation_mask))
diagnostic_scores = diagnostic.select('MKT_score', 'SIZE_score', 'GV_score').to_numpy()
max_score = np.nanmax(diagnostic_scores, axis=1)
diagnostic = diagnostic.with_columns(pl.Series('max_factor_score', max_score), pl.Series('factor_alert', max_score > 1))
threshold_summary = pl.DataFrame([{
    'shock_abs_error_threshold_pct': shock_threshold * 100,
    'MKT_product_threshold': factor_thresholds[0], 'SIZE_product_threshold': factor_thresholds[1], 'GV_product_threshold': factor_thresholds[2],
}]).with_columns(pl.all().round(8))
threshold_summary

shock_abs_error_threshold_pct,MKT_product_threshold,SIZE_product_threshold,GV_product_threshold
f64,f64,f64,f64
4.551398,0.000154,0.000191,0.000128


## <b>Episodios candidatos y adaptación</b>

> Un candidato inicia en la primera fecha de un bloque de alertas factoriales. Los bloques se ordenan por su máxima intensidad durante las 21 sesiones siguientes, se mantienen separados por al menos 63 sesiones y requieren 63 sesiones posteriores disponibles. No se seleccionan por la respuesta posterior de Kalman.

In [4]:
alerts = diagnostic.get_column('factor_alert').to_numpy()
candidate_starts = []
for index in range(diagnostic.height - 63):
    if alerts[index] and (index == 0 or not alerts[index - 1]):
        candidate_starts.append((index, diagnostic.get_column('max_factor_score')[index:index + 21].max()))
ordered = sorted(candidate_starts, key=lambda item: item[1], reverse=True)
episode_index = []
for index, _ in ordered:
    if all(abs(index - prior) >= 63 for prior in episode_index):
        episode_index.append(index)
    if len(episode_index) == 3:
        break
episode_index.sort()
factor_labels = np.asarray(['MKT', 'SIZE', 'GV'])
xle_kalman = kalman.filter(pl.col('asset') == 'XLE').sort('date')
episode_rows, recovery_rows = [], []
for number, index in enumerate(episode_index, start=1):
    row = diagnostic.row(index, named=True)
    scores = np.asarray([row['MKT_score'], row['SIZE_score'], row['GV_score']])
    main_factor = factor_labels[np.nanargmax(scores)]
    post21 = diagnostic.slice(index, min(21, diagnostic.height - index))
    post63 = diagnostic.slice(index, min(63, diagnostic.height - index))
    ols_beta = xle_ols_all.filter(pl.col('date') == row['date']).select('beta_market', 'beta_size', 'beta_growth_value').row(0)
    kalman_beta = xle_kalman.filter(pl.col('date') == row['date']).select('beta_market', 'beta_size', 'beta_growth_value').row(0)
    episode_rows.append({'episode': number, 'first_alert_date': row['date'], 'main_factor': main_factor,
                         'ols_error_pct': row['residual_return'] * 100, 'max_factor_score': row['max_factor_score'],
                         'factor_alerts_next21': int(post21.get_column('factor_alert').sum()),
                         'shock_alerts_next21': int(post21.get_column('shock_alert').sum()),
                         'beta_gap_MKT': ols_beta[0] - kalman_beta[0], 'beta_gap_SIZE': ols_beta[1] - kalman_beta[1],
                         'beta_gap_GV': ols_beta[2] - kalman_beta[2]})
    for model, frame in (('Rolling OLS', xle_ols_all.filter(pl.col('date') >= EVALUATION_START).sort('date')), ('Kalman fixed R', xle_kalman)):
        position = frame.get_column('date').to_list().index(row['date'])
        before = frame.slice(max(0, position - 21), min(21, position)).get_column('residual_return').to_numpy()
        after21 = frame.slice(position, min(21, frame.height - position)).get_column('residual_return').to_numpy()
        after63 = frame.slice(position, min(63, frame.height - position)).get_column('residual_return').to_numpy()
        recovery_rows.append({'episode': number, 'model': model, 'mae_before21_pct': np.abs(before).mean() * 100,
                              'rmse_before21_pct': np.sqrt(np.mean(before ** 2)) * 100,
                              'mae_after21_pct': np.abs(after21).mean() * 100, 'rmse_after21_pct': np.sqrt(np.mean(after21 ** 2)) * 100,
                              'cum_error_21_pct': after21.sum() * 100, 'cum_error_63_pct': after63.sum() * 100})
episodes = pl.DataFrame(episode_rows).with_columns(pl.all().exclude('episode', 'first_alert_date', 'main_factor', 'factor_alerts_next21', 'shock_alerts_next21').round(3))
recovery = pl.DataFrame(recovery_rows).with_columns(pl.all().exclude('episode', 'model').round(3))
episodes, recovery

(shape: (3, 10)
 ┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
 │ episode ┆ first_aler ┆ main_fact ┆ ols_error ┆ … ┆ shock_ale ┆ beta_gap_ ┆ beta_gap_ ┆ beta_gap_ │
 │ ---     ┆ t_date     ┆ or        ┆ _pct      ┆   ┆ rts_next2 ┆ MKT       ┆ SIZE      ┆ GV        │
 │ i64     ┆ ---        ┆ ---       ┆ ---       ┆   ┆ 1         ┆ ---       ┆ ---       ┆ ---       │
 │         ┆ date       ┆ str       ┆ f64       ┆   ┆ ---       ┆ f64       ┆ f64       ┆ f64       │
 │         ┆            ┆           ┆           ┆   ┆ i64       ┆           ┆           ┆           │
 ╞═════════╪════════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
 │ 1       ┆ 2022-03-07 ┆ MKT       ┆ 2.708     ┆ … ┆ 0         ┆ 0.05      ┆ 0.048     ┆ 0.075     │
 │ 2       ┆ 2025-04-04 ┆ MKT       ┆ -3.373    ┆ … ┆ 0         ┆ -0.085    ┆ -0.062    ┆ -0.074    │
 │ 3       ┆ 2026-04-08 ┆ MKT       ┆ -5.983    ┆ … ┆ 1         ┆ 

## <b>Visualización de diagnósticos</b>

La primera figura muestra los productos factor–error escalados por sus umbrales previos a 2022. La segunda alinea errores y diferencias de beta alrededor de los candidatos seleccionados mecánicamente.

In [6]:
figure = go.Figure()
for column, label in (('MKT_score', 'MKT'), ('SIZE_score', 'SIZE'), ('GV_score', 'GV')):
    figure.add_scatter(x=diagnostic['date'], y=diagnostic[column], mode='lines', name=label)
figure.add_hline(y=1, line_dash='dash', line_color='gray', annotation_text='historical threshold')
for index in episode_index:
    figure.add_vline(x=diagnostic.get_column('date')[index], line_dash='dot', line_color='black')
figure.update_layout(title='XLE — factor-error instability diagnostic', xaxis_title='return date', yaxis_title='absolute score / historical threshold', template='plotly_white')
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))



In [7]:
figure = make_subplots(rows=2, cols=len(episode_index), shared_yaxes='rows', subplot_titles=[str(diagnostic.get_column('date')[index]) for index in episode_index],
                       vertical_spacing=.16, row_titles=['attribution error (%)', 'OLS beta minus Kalman beta'])
for column, center in enumerate(episode_index, start=1):
    start, end = max(0, center - 21), min(diagnostic.height, center + 64)
    dates_window = diagnostic.get_column('date')[start:end]
    ols_window = xle_ols_all.filter(pl.col('date').is_in(dates_window)).sort('date')
    kalman_window = xle_kalman.filter(pl.col('date').is_in(dates_window)).sort('date')
    for label, frame, color in (('Rolling OLS', ols_window, '#636EFA'), ('Kalman', kalman_window, '#EF553B')):
        figure.add_scatter(x=frame['date'], y=frame['residual_return'] * 100, mode='lines', name=label, line={'color': color},
                           legendgroup=label, showlegend=column == 1, row=1, col=column)
    for beta, label, color in (('beta_market', 'MKT gap', '#00CC96'), ('beta_size', 'SIZE gap', '#AB63FA'), ('beta_growth_value', 'GV gap', '#FFA15A')):
        figure.add_scatter(x=ols_window['date'], y=ols_window[beta] - kalman_window[beta], mode='lines', name=label, line={'color': color},
                           legendgroup=label, showlegend=column == 1, row=2, col=column)
figure.update_layout(title='XLE — model response around factor-instability candidates', template='plotly_white', height=600)
display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))

C:\Users\avazq\AppData\Local\Temp\ipykernel_18360\2034399268.py:6: DeprecationWarning:

`is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.

C:\Users\avazq\AppData\Local\Temp\ipykernel_18360\2034399268.py:7: DeprecationWarning:

`is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.



## <b>Conclusión</b>

- Los tres candidatos de XLE inician en alertas MKT y mantienen 16–21 alertas factoriales en las 21 sesiones posteriores. La evidencia apunta a relación persistente entre error y mercado, no sólo a errores diarios aislados.
- Sólo el episodio de 2026 contiene un shock residual que supera el umbral histórico de 4.55%; los otros dos persisten sin excederlo. Magnitud diaria y relación factor–error aportan diagnósticos distintos.
- Kalman reduce MAE y RMSE posteriores en los episodios de 2022 y 2026, y reduce el error acumulado de 63 sesiones en 2026. En 2025 es peor que OLS; no hay adaptación superior uniforme.
- Las alertas repetidas y su asociación con MKT justifican estudiar un detector explícito pequeño, pero no validar una ruptura estructural ni modificar los modelos congelados.
- No se identifica de forma única beta drift: las alertas también pueden reflejar factores omitidos, cambios en la composición de XLE o dependencia transitoria. La muestra ya fue examinada y los umbrales no son pruebas confirmatorias ni señales de trading.